In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# [셀 0] 기본 import
from pathlib import Path
import random
import shutil
import json
import yaml
import sys
import importlib
from collections import defaultdict, Counter
from tqdm.auto import tqdm

In [11]:
# [셀 1] 경로 설정 + split 설정

BASE_DIR = Path("/content/drive/MyDrive/먼작귀")
SYNTH_DIR = BASE_DIR / "dataset" / "synthetic"
OUT_DIR = BASE_DIR / "dataset" / "yolo_scenario_dataset"

SCENARIO_COUNTS = {
    "s0": 280,
    "s1": 200,
    "s2": 180,
    "s3": 200,
    "s4": 160,
    "s5": 240,
    "s6": 200,
    "s7": 140,
}

VIEWS = ["front", "side"]

LABEL_VIEW_BY_VIEW = {
    "front": "front",
    "side": "side_train",
}

SPLIT_RATIO = {
    "train": 0.7,
    "val": 0.2,
    "test": 0.1,
}

RANDOM_SEED = 42

print("SYNTH_DIR:", SYNTH_DIR, SYNTH_DIR.exists())
print("OUT_DIR:", OUT_DIR)
print("LABEL_VIEW_BY_VIEW:", LABEL_VIEW_BY_VIEW)

# 기존 yolo_scenario_dataset 삭제 후 새로 생성
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
    print("기존 OUT_DIR 삭제:", OUT_DIR)

for split in ["train", "val", "test"]:
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("폴더 생성 완료")
for split in ["train", "val", "test"]:
    print(OUT_DIR / "images" / split, "exists:", (OUT_DIR / "images" / split).exists())
    print(OUT_DIR / "labels" / split, "exists:", (OUT_DIR / "labels" / split).exists())


SYNTH_DIR: /content/drive/MyDrive/먼작귀/dataset/synthetic True
OUT_DIR: /content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset
LABEL_VIEW_BY_VIEW: {'front': 'front', 'side': 'side_train'}
폴더 생성 완료
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/images/train exists: True
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/labels/train exists: True
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/images/val exists: True
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/labels/val exists: True
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/images/test exists: True
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/labels/test exists: True


In [12]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/먼작귀")
SYNTH_DIR = BASE_DIR / "dataset" / "synthetic"

print("BASE_DIR 존재:", BASE_DIR.exists(), BASE_DIR)
print("SYNTH_DIR 존재:", SYNTH_DIR.exists(), SYNTH_DIR)

print("\n===== dataset 폴더 안 =====")
for p in sorted((BASE_DIR / "dataset").glob("*")):
    print(p)

print("\n===== synthetic 폴더 안 =====")
if SYNTH_DIR.exists():
    for p in sorted(SYNTH_DIR.glob("*")):
        print(p)

BASE_DIR 존재: True /content/drive/MyDrive/먼작귀
SYNTH_DIR 존재: True /content/drive/MyDrive/먼작귀/dataset/synthetic

===== dataset 폴더 안 =====
/content/drive/MyDrive/먼작귀/dataset/bg_removed
/content/drive/MyDrive/먼작귀/dataset/item_list
/content/drive/MyDrive/먼작귀/dataset/pairs_classnames_cache.pkl
/content/drive/MyDrive/먼작귀/dataset/shelf_lip_yolo_seg
/content/drive/MyDrive/먼작귀/dataset/synthetic
/content/drive/MyDrive/먼작귀/dataset/training
/content/drive/MyDrive/먼작귀/dataset/training_final
/content/drive/MyDrive/먼작귀/dataset/yolo_product_dataset
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset

===== synthetic 폴더 안 =====
/content/drive/MyDrive/먼작귀/dataset/synthetic/images
/content/drive/MyDrive/먼작귀/dataset/synthetic/labels_json
/content/drive/MyDrive/먼작귀/dataset/synthetic/labels_yolo
/content/drive/MyDrive/먼작귀/dataset/synthetic/labels_yolo_seg


In [13]:
# [셀 2] source 이미지/라벨 폴더 사전 검수

precheck_errors = []

for view in VIEWS:
    label_view = LABEL_VIEW_BY_VIEW[view]

    for scenario, expected_count in SCENARIO_COUNTS.items():
        image_dir = SYNTH_DIR / "images" / view / scenario
        label_dir = SYNTH_DIR / "labels_yolo" / label_view / scenario

        image_stems = {p.stem for p in image_dir.glob("*.png")} if image_dir.exists() else set()
        label_stems = {p.stem for p in label_dir.glob("*.txt")} if label_dir.exists() else set()

        missing_label = sorted(image_stems - label_stems)
        extra_label = sorted(label_stems - image_stems)

        print(f"[{view}/{scenario}] image -> labels_yolo/{label_view}/{scenario}")
        print("image_dir:", image_dir, image_dir.exists())
        print("label_dir:", label_dir, label_dir.exists())
        print("image 수:", len(image_stems), "| label 수:", len(label_stems), "| expected:", expected_count)
        print("라벨 없는 이미지 수:", len(missing_label))
        print("이미지 없는 라벨 수:", len(extra_label))

        if len(image_stems) != expected_count:
            precheck_errors.append((view, scenario, "image_count_mismatch", expected_count, len(image_stems)))

        if not image_dir.exists():
            precheck_errors.append((view, scenario, "missing_image_dir", str(image_dir)))

        if not label_dir.exists():
            precheck_errors.append((view, scenario, "missing_label_dir", str(label_dir)))

        if missing_label:
            precheck_errors.append((view, scenario, "missing_label", missing_label[:10]))
            print("라벨 없는 이미지 예시:", missing_label[:10])

        if extra_label:
            precheck_errors.append((view, scenario, "extra_label", extra_label[:10]))
            print("이미지 없는 라벨 예시:", extra_label[:10])

print("===== 사전 검수 완료 =====")
print("문제 수:", len(precheck_errors))

if precheck_errors:
    print("문제 예시:")
    for x in precheck_errors[:20]:
        print(x)
    raise ValueError("source image/label 매칭 문제가 있습니다. 위 문제를 먼저 해결하세요.")


[front/s0] image -> labels_yolo/front/s0
image_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/images/front/s0 True
label_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/labels_yolo/front/s0 True
image 수: 280 | label 수: 280 | expected: 280
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
[front/s1] image -> labels_yolo/front/s1
image_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/images/front/s1 True
label_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/labels_yolo/front/s1 True
image 수: 200 | label 수: 200 | expected: 200
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
[front/s2] image -> labels_yolo/front/s2
image_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/images/front/s2 True
label_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/labels_yolo/front/s2 True
image 수: 180 | label 수: 180 | expected: 180
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
[front/s3] image -> labels_yolo/front/s3
image_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/images/front/s3 True
label_dir: /content/drive/MyDrive/먼작귀/dataset/synthetic/labe

In [14]:
# [셀 3] 시나리오별 70/20/10 split + 복사

def split_files_stratified(image_paths, seed=42):
    image_paths = list(image_paths)
    random.Random(seed).shuffle(image_paths)

    n = len(image_paths)
    n_train = int(n * SPLIT_RATIO["train"])
    n_val = int(n * SPLIT_RATIO["val"])

    train_files = image_paths[:n_train]
    val_files = image_paths[n_train:n_train + n_val]
    test_files = image_paths[n_train + n_val:]

    return {
        "train": train_files,
        "val": val_files,
        "test": test_files,
    }

copy_logs = []
missing_pairs = []
split_summary = defaultdict(lambda: defaultdict(int))

for view in VIEWS:
    label_view = LABEL_VIEW_BY_VIEW[view]

    for scenario, expected_count in SCENARIO_COUNTS.items():
        image_dir = SYNTH_DIR / "images" / view / scenario
        label_dir = SYNTH_DIR / "labels_yolo" / label_view / scenario

        image_paths = sorted(image_dir.glob("*.png"))

        split_map = split_files_stratified(
            image_paths=image_paths,
            seed=RANDOM_SEED
        )

        for split, paths in split_map.items():
            for img_path in tqdm(paths, desc=f"copy {view}/{scenario}/{split}", leave=False):
                label_path = label_dir / f"{img_path.stem}.txt"

                if not label_path.exists():
                    missing_pairs.append({
                        "view": view,
                        "label_view": label_view,
                        "scenario": scenario,
                        "image": str(img_path),
                        "missing_label": str(label_path),
                    })
                    continue

                # 파일명 충돌 방지: view + scenario prefix 붙임
                new_stem = f"{view}_{scenario}_{img_path.stem}"

                out_img_path = OUT_DIR / "images" / split / f"{new_stem}.png"
                out_label_path = OUT_DIR / "labels" / split / f"{new_stem}.txt"

                shutil.copy2(img_path, out_img_path)
                shutil.copy2(label_path, out_label_path)

                split_summary[f"{view}/{scenario}"][split] += 1
                copy_logs.append({
                    "view": view,
                    "label_view": label_view,
                    "scenario": scenario,
                    "split": split,
                    "src_image": str(img_path),
                    "src_label": str(label_path),
                    "out_image": str(out_img_path),
                    "out_label": str(out_label_path),
                })

print("복사 완료")
print("복사된 pair 수:", len(copy_logs))
print("missing pair 수:", len(missing_pairs))

if missing_pairs:
    print("missing pair 예시:")
    for x in missing_pairs[:20]:
        print(x)
    raise ValueError("라벨 누락이 있습니다.")


copy front/s0/train:   0%|          | 0/196 [00:00<?, ?it/s]

copy front/s0/val:   0%|          | 0/56 [00:00<?, ?it/s]

copy front/s0/test:   0%|          | 0/28 [00:00<?, ?it/s]

copy front/s1/train:   0%|          | 0/140 [00:00<?, ?it/s]

copy front/s1/val:   0%|          | 0/40 [00:00<?, ?it/s]

...

copy side/s6/test:   0%|          | 0/20 [00:00<?, ?it/s]

copy side/s7/train:   0%|          | 0/98 [00:00<?, ?it/s]

copy side/s7/val:   0%|          | 0/28 [00:00<?, ?it/s]

copy side/s7/test:   0%|          | 0/14 [00:00<?, ?it/s]

복사 완료
복사된 pair 수: 3200
missing pair 수: 0


In [16]:
# [셀 4] split 결과 확인
print("===== Split Summary =====")

total_by_split = defaultdict(int)

for key in sorted(split_summary.keys()):
    train_n = split_summary[key]["train"]
    val_n = split_summary[key]["val"]
    test_n = split_summary[key]["test"]
    total = train_n + val_n + test_n

    total_by_split["train"] += train_n
    total_by_split["val"] += val_n
    total_by_split["test"] += test_n

    print(
        f"{key:10s} | "
        f"train={train_n:4d} | val={val_n:4d} | test={test_n:4d} | total={total:4d}"
    )

print("===== Total =====")
print("train:", total_by_split["train"])
print("val:", total_by_split["val"])
print("test:", total_by_split["test"])
print("all:", sum(total_by_split.values()))

print("실제 폴더 파일 수:")
for split in ["train", "val", "test"]:
    img_count = len(list((OUT_DIR / "images" / split).glob("*.png")))
    label_count = len(list((OUT_DIR / "labels" / split).glob("*.txt")))

    print(split, "| images:", img_count, "| labels:", label_count)


===== Split Summary =====
front/s0   | train= 196 | val=  56 | test=  28 | total= 280
front/s1   | train= 140 | val=  40 | test=  20 | total= 200
front/s2   | train= 125 | val=  36 | test=  19 | total= 180
front/s3   | train= 140 | val=  40 | test=  20 | total= 200
front/s4   | train= 112 | val=  32 | test=  16 | total= 160
front/s5   | train= 168 | val=  48 | test=  24 | total= 240
front/s6   | train= 140 | val=  40 | test=  20 | total= 200
front/s7   | train=  98 | val=  28 | test=  14 | total= 140
side/s0    | train= 196 | val=  56 | test=  28 | total= 280
side/s1    | train= 140 | val=  40 | test=  20 | total= 200
side/s2    | train= 125 | val=  36 | test=  19 | total= 180
side/s3    | train= 140 | val=  40 | test=  20 | total= 200
side/s4    | train= 112 | val=  32 | test=  16 | total= 160
side/s5    | train= 168 | val=  48 | test=  24 | total= 240
side/s6    | train= 140 | val=  40 | test=  20 | total= 200
side/s7    | train=  98 | val=  28 | test=  14 | total= 140
===== Total ==

In [17]:
# [셀 5] 최종 yolo_scenario_dataset image-label 1:1 검수
pair_errors = []

for split in ["train", "val", "test"]:
    image_dir = OUT_DIR / "images" / split
    label_dir = OUT_DIR / "labels" / split

    image_stems = {p.stem for p in image_dir.glob("*.png")}
    label_stems = {p.stem for p in label_dir.glob("*.txt")}

    missing_label = sorted(image_stems - label_stems)
    extra_label = sorted(label_stems - image_stems)

    print(f"[{split}]")
    print("image 수:", len(image_stems))
    print("label 수:", len(label_stems))
    print("라벨 없는 이미지 수:", len(missing_label))
    print("이미지 없는 라벨 수:", len(extra_label))

    if missing_label:
        pair_errors.append((split, "missing_label", missing_label[:10]))
        print("라벨 없는 이미지 예시:", missing_label[:10])

    if extra_label:
        pair_errors.append((split, "extra_label", extra_label[:10]))
        print("이미지 없는 라벨 예시:", extra_label[:10])

if pair_errors:
    raise ValueError("최종 dataset image-label 매칭 문제가 있습니다.")
else:
    print("최종 image-label 1:1 매칭 정상")


[train]
image 수: 2238
label 수: 2238
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
[val]
image 수: 640
label 수: 640
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
[test]
image 수: 322
label 수: 322
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
최종 image-label 1:1 매칭 정상


In [18]:
# [셀 6] class_id / bbox 형식 검수
class_counter = Counter()
problem_lines = []
empty_label_files = []

for label_path in (OUT_DIR / "labels").rglob("*.txt"):
    text = label_path.read_text(encoding="utf-8").strip()

    if not text:
        empty_label_files.append(str(label_path))
        continue

    for line_no, line in enumerate(text.splitlines(), start=1):
        parts = line.strip().split()

        if len(parts) != 5:
            problem_lines.append((str(label_path), line_no, line, "컬럼 수 오류"))
            continue

        try:
            cid = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except Exception as e:
            problem_lines.append((str(label_path), line_no, line, f"파싱 오류: {e}"))
            continue

        if cid < 0 or cid > 66:
            problem_lines.append((str(label_path), line_no, line, "class_id 범위 오류"))
            continue

        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            problem_lines.append((str(label_path), line_no, line, "bbox 범위 오류"))
            continue

        class_counter[cid] += 1

class_ids = sorted(class_counter.keys())

print("===== class/bbox 검수 =====")
print("전체 객체 수:", sum(class_counter.values()))
print("사용된 class_id 개수:", len(class_ids))
print("최소 class_id:", min(class_ids) if class_ids else None)
print("최대 class_id:", max(class_ids) if class_ids else None)
print("빈 라벨 파일 수:", len(empty_label_files))
print("문제 line 수:", len(problem_lines))
print("사용 class_id 목록:")
print(class_ids)

if problem_lines:
    print("문제 line 예시:")
    for x in problem_lines[:20]:
        print(x)
    raise ValueError("라벨 txt 형식 문제가 있습니다.")

if empty_label_files:
    print("빈 라벨 파일 예시:")
    for x in empty_label_files[:10]:
        print(x)


===== class/bbox 검수 =====
전체 객체 수: 305585
사용된 class_id 개수: 61
최소 class_id: 0
최대 class_id: 66
빈 라벨 파일 수: 0
문제 line 수: 0
사용 class_id 목록:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 24, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66]


In [19]:
# [셀 7] data.yaml 생성

COMMON_DIR = BASE_DIR / "synthetic_scenarios"
if str(COMMON_DIR) not in sys.path:
    sys.path.insert(0, str(COMMON_DIR))

import product_class_map as classmap
importlib.reload(classmap)

print("class map 검수:", classmap.validate_class_map())

names = classmap.PRODUCT_CLASS_NAMES

# Ultralytics data.yaml
# names는 {0: 상품명, 1: 상품명, ...} dict 형태로 저장해도 정상 인식됨
data_yaml = {
    "path": str(OUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": names,
}

yaml_path = OUT_DIR / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_yaml,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("data.yaml 저장 완료:", yaml_path)
print("data.yaml class 수:", len(names))
print(yaml_path.read_text(encoding="utf-8")[:2000])


class map 검수: {'num_classes': 67, 'min_class_id': 0, 'max_class_id': 66, 'num_name_map': 67}
data.yaml 저장 완료: /content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/data.yaml
data.yaml class 수: 67
path: /content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset
train: images/train
val: images/val
test: images/test
names:
  0: (주)국모싸이언스)메디안칼슘치약75G
  1: CJ다담떡볶이양념
  2: CJ렛츠웰맛밤80G
  3: CJ맥스봉체다치즈어랏400G
  4: CJ비비고스팸부대찌개460G
  5: CJ스팸200G
  6: CJ햇반컵반설렁탕밥253g
  7: 골드)황도슬라이스
  8: 길림양행)구운아몬드
  9: 깨끗한나라여행용티슈핑크50매
  10: 꼬깔콘고소한맛72G
  11: 농심)사리곰탕110G(봉지)
  12: 농심)신라면건면(낱개)97G
  13: 농심)프링글스클래식110G
  14: 농심매운새우깡90G
  15: 농심보노콘스프3입18.6g
  16: 농심새우탕컵(소)67G
  17: 농심순한너구리120G
  18: 농심양파링84G
  19: 농심짜파게티범벅70G
  20: 농심짜파게티큰사발123G
  21: 델몬트후레쉬컷슬라이스파인애플836G
  22: 동서맥심카누마일드로스트스위트아메리카노10T
  23: 동아제약)가그린후레쉬라임100ML
  24: 동원)고추참치
  25: 동원살코기참치250G
  26: 롯데런천미트340G
  27: 롯데빈츠204G
  28: 롯데칸쵸컵88G
  29: 리뉴후래쉬60ML-1118입고
  30: 머거본)콘소메맛아몬드
  31: 명성식품)한입부산어포
  32: 목우촌육포
  33: 미소)초이스엘참치_닭가슴살5개입
  34: 삼양)불닭소스200G
  35: 삼양크

In [ ]:
# [셀 8] 최종 확인
print("OUT_DIR:", OUT_DIR)
print("data.yaml:", OUT_DIR / "data.yaml", (OUT_DIR / "data.yaml").exists())

for split in ["train", "val", "test"]:
    print(
        split,
        "images:", len(list((OUT_DIR / "images" / split).glob("*.png"))),
        "labels:", len(list((OUT_DIR / "labels" / split).glob("*.txt"))),
    )

print("학습 시 사용 경로:")
print(str(OUT_DIR / "data.yaml"))

OUT_DIR: /content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset
data.yaml: /content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/data.yaml True
train images: 0 labels: 0
val images: 0 labels: 0
test images: 0 labels: 0
학습 시 사용 경로:
/content/drive/MyDrive/먼작귀/dataset/yolo_scenario_dataset/data.yaml
